# Convert Mayo Analyze `.obj` ROI Object Maps to NIfTI masks

This notebook converts masks exported from **Mayo Analyze / Analyze object maps** into NIfTI mask volumes that can be used in Python, MONAI, napari, ITK-SNAP, or downstream AI training.

Your pasted file looked like a binary Analyze Object Map, not a Wavefront mesh. The practical conversion route is:

```text
Analyze .obj object map
        -> decoded 3D label volume
        -> NIfTI .nii.gz mask using the matching MRI as spatial reference
```

## What this notebook supports

1. **Analyze binary Object Map `.obj`** using MATLAB `decodeOBJ3.m` or `decodeOBJ.m`.
2. **Wavefront mesh `.obj`** fallback, in case another export is a real mesh.
3. Batch conversion from many `.obj` files to `.nii.gz` masks.
4. QC overlays to verify the mask lines up with the MRI.

## What you need

For each mask, you need:

- the original Analyze `.obj` file, not a copy-pasted text version
- the matching MRI volume, usually `.nii.gz`, `.nii`, or Analyze `.hdr/.img`
- MATLAB installed, plus `decodeOBJ3.m` or `decodeOBJ.m` on disk

Recommended reader:

- `decodeOBJ3(file)` from MATLAB File Exchange for Analyze version 7 object maps.
- `decodeOBJ(file)` for older Analyze 6 object maps.

## Conda environment setup

Run this in a terminal, preferably inside VS Code:

```bash
conda create -n analyze_obj_convert python=3.11 -y
conda activate analyze_obj_convert
conda install -c conda-forge numpy scipy nibabel matplotlib pandas trimesh scikit-image ipykernel -y
python -m ipykernel install --user --name analyze_obj_convert --display-name "Python (analyze_obj_convert)"
```

Then restart VS Code and choose the kernel **Python (analyze_obj_convert)**.

MATLAB support options:

- The easiest method is for this notebook to call MATLAB from the command line using `matlab -batch`.
- You do **not** need the MATLAB Python engine for the default path.
- You do need MATLAB installed and available on your system path, or you need to set `MATLAB_EXE` below.

In [ ]:
from pathlib import Path
import os
import re
import json
import shutil
import subprocess
import tempfile
import textwrap

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy import ndimage as ndi

print("Imports complete")

# 1. Configuration

Edit these paths first.

Important:

- `OBJ_PATH` should point to the original binary `.obj` file exported from Analyze.
- `REFERENCE_MRI_PATH` should point to the matching MRI volume.
- `DECODEOBJ_M_DIR` should be the folder containing `decodeOBJ3.m` or `decodeOBJ.m`.

In [ ]:
from pathlib import Path
import shutil

# ============================================================
# EDIT ONLY THIS FOLDER
# ============================================================
SEARCH_FOLDER = Path(r"Z:\path\to\folder_containing_everything")

# Optional: leave as None unless MATLAB is not on your PATH.
# Example:
# MATLAB_EXE = r"C:\Program Files\MATLAB\R2024b\bin\matlab.exe"
MATLAB_EXE = None

# Optional: choose "decodeOBJ3", "decodeOBJ", or "auto"
DECODE_FUNCTION = "auto"

# Output folder will be created inside SEARCH_FOLDER
OUTPUT_DIR = SEARCH_FOLDER / "converted_masks"

# ============================================================
# AUTO-FIND FILES INSIDE SEARCH_FOLDER
# ============================================================

if not SEARCH_FOLDER.exists():
    raise FileNotFoundError(f"SEARCH_FOLDER does not exist:\n{SEARCH_FOLDER}")

if not SEARCH_FOLDER.is_dir():
    raise NotADirectoryError(f"SEARCH_FOLDER is not a folder:\n{SEARCH_FOLDER}")

def is_probably_wavefront_obj(path, bytes_to_read=1024):
    """
    A normal Wavefront mesh OBJ is plain text and usually contains lines
    beginning with v, vt, vn, f, mtllib, or usemtl.
    Analyze object maps are binary and should not look like this.
    """
    try:
        raw = path.read_bytes()[:bytes_to_read]
    except Exception:
        return False

    text = raw.decode("utf-8", errors="ignore").lower()

    wavefront_markers = [
        "\nv ",
        "\nf ",
        "\nvt ",
        "\nvn ",
        "mtllib",
        "usemtl",
    ]

    return any(marker in text for marker in wavefront_markers)


def is_probably_analyze_object_map(path, bytes_to_read=8192):
    """
    Heuristic for Mayo Analyze binary object map files.
    The pasted example had embedded strings like:
    Original, Object_2, 1.Object, 2.Object
    """
    try:
        raw = path.read_bytes()[:bytes_to_read]
    except Exception:
        return False

    has_null_bytes = b"\x00" in raw
    has_object_tokens = any(
        token in raw
        for token in [b"Original", b"Object", b".Object"]
    )

    return has_null_bytes and has_object_tokens and not is_probably_wavefront_obj(path)


def file_size_mb(path):
    try:
        return path.stat().st_size / (1024 ** 2)
    except Exception:
        return 0


def score_obj(path):
    """
    Higher score means more likely to be the Analyze ROI object map.
    """
    name = path.name.lower()
    score = 0

    if path.suffix.lower() == ".obj":
        score += 20

    if is_probably_analyze_object_map(path):
        score += 100

    if is_probably_wavefront_obj(path):
        score -= 100

    for word in ["mask", "roi", "object", "seg", "label", "tumor"]:
        if word in name:
            score += 5

    for word in ["mesh", "surface", "model"]:
        if word in name:
            score -= 20

    return score


def score_mri(path, obj_path=None):
    """
    Higher score means more likely to be the matching reference MRI.
    """
    name = path.name.lower()
    score = 0

    if name.endswith(".nii.gz"):
        score += 40
    elif path.suffix.lower() == ".nii":
        score += 35
    elif path.suffix.lower() in [".hdr", ".img"]:
        score += 20

    for word in ["mri", "brain", "mouse", "scan", "t1", "t2", "rare", "gre", "anat"]:
        if word in name:
            score += 5

    for word in ["mask", "seg", "label", "roi", "object", "tumor"]:
        if word in name:
            score -= 25

    # Prefer larger files for MRI volumes
    score += min(file_size_mb(path), 500) / 50

    # Prefer files in same folder as OBJ
    if obj_path is not None:
        if path.parent == obj_path.parent:
            score += 20

        obj_words = set(obj_path.stem.lower().replace("-", "_").split("_"))
        mri_words = set(path.stem.lower().replace("-", "_").split("_"))
        shared = obj_words.intersection(mri_words)
        score += len(shared) * 3

    return score


# -----------------------------
# Search recursively inside only SEARCH_FOLDER
# -----------------------------
obj_candidates = sorted(
    [p for p in SEARCH_FOLDER.rglob("*.obj") if p.is_file()],
    key=score_obj,
    reverse=True,
)

mri_candidates = sorted(
    [
        p for p in SEARCH_FOLDER.rglob("*")
        if p.is_file()
        and (
            p.name.lower().endswith(".nii.gz")
            or p.suffix.lower() in [".nii", ".hdr", ".img"]
        )
    ],
    key=lambda p: score_mri(p, obj_candidates[0] if obj_candidates else None),
    reverse=True,
)

decodeobj3_candidates = sorted(
    [p for p in SEARCH_FOLDER.rglob("decodeOBJ3.m") if p.is_file()]
)

decodeobj_candidates = sorted(
    [p for p in SEARCH_FOLDER.rglob("decodeOBJ.m") if p.is_file()]
)

# -----------------------------
# Choose OBJ
# -----------------------------
if not obj_candidates:
    raise FileNotFoundError(
        f"No .obj files were found inside:\n{SEARCH_FOLDER}"
    )

OBJ_PATH = obj_candidates[0]

# -----------------------------
# Choose MRI
# -----------------------------
if not mri_candidates:
    raise FileNotFoundError(
        f"No reference MRI files were found inside:\n{SEARCH_FOLDER}\n\n"
        "Expected one of: .nii.gz, .nii, .hdr, .img"
    )

REFERENCE_MRI_PATH = mri_candidates[0]

# -----------------------------
# Choose decode function and folder
# -----------------------------
if DECODE_FUNCTION == "auto":
    if decodeobj3_candidates:
        DECODE_FUNCTION = "decodeOBJ3"
        DECODEOBJ_M_DIR = decodeobj3_candidates[0].parent
    elif decodeobj_candidates:
        DECODE_FUNCTION = "decodeOBJ"
        DECODEOBJ_M_DIR = decodeobj_candidates[0].parent
    else:
        raise FileNotFoundError(
            f"Could not find decodeOBJ3.m or decodeOBJ.m inside:\n{SEARCH_FOLDER}\n\n"
            "Put decodeOBJ3.m or decodeOBJ.m somewhere inside this folder and rerun."
        )
elif DECODE_FUNCTION == "decodeOBJ3":
    if not decodeobj3_candidates:
        raise FileNotFoundError(
            f"DECODE_FUNCTION is set to decodeOBJ3, but decodeOBJ3.m was not found inside:\n{SEARCH_FOLDER}"
        )
    DECODEOBJ_M_DIR = decodeobj3_candidates[0].parent
elif DECODE_FUNCTION == "decodeOBJ":
    if not decodeobj_candidates:
        raise FileNotFoundError(
            f"DECODE_FUNCTION is set to decodeOBJ, but decodeOBJ.m was not found inside:\n{SEARCH_FOLDER}"
        )
    DECODEOBJ_M_DIR = decodeobj_candidates[0].parent
else:
    raise ValueError(
        'DECODE_FUNCTION must be "auto", "decodeOBJ3", or "decodeOBJ".'
    )

# -----------------------------
# Find MATLAB executable if possible
# -----------------------------
if MATLAB_EXE is None:
    MATLAB_EXE = shutil.which("matlab")

# -----------------------------
# Output files
# -----------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MAT_PATH = OUTPUT_DIR / (OBJ_PATH.stem + "_decoded_from_matlab.mat")
OUT_NIFTI_PATH = OUTPUT_DIR / (OBJ_PATH.stem + "_mask.nii.gz")
OUT_QC_PNG = OUTPUT_DIR / (OBJ_PATH.stem + "_qc_overlay.png")

# -----------------------------
# Print results and alternatives
# -----------------------------
print("Selected files:")
print("OBJ_PATH:          ", OBJ_PATH)
print("REFERENCE_MRI_PATH:", REFERENCE_MRI_PATH)
print("DECODEOBJ_M_DIR:   ", DECODEOBJ_M_DIR)
print("DECODE_FUNCTION:   ", DECODE_FUNCTION)
print("MATLAB_EXE:        ", MATLAB_EXE)
print("OUTPUT_DIR:        ", OUTPUT_DIR)
print("OUT_MAT_PATH:      ", OUT_MAT_PATH)
print("OUT_NIFTI_PATH:    ", OUT_NIFTI_PATH)
print("OUT_QC_PNG:        ", OUT_QC_PNG)

print("\nOBJ candidates:")
for p in obj_candidates[:10]:
    print(f"  score={score_obj(p):>6.1f}  size={file_size_mb(p):>8.2f} MB  {p}")

print("\nMRI candidates:")
for p in mri_candidates[:10]:
    print(f"  score={score_mri(p, OBJ_PATH):>6.1f}  size={file_size_mb(p):>8.2f} MB  {p}")

print("\nDecode candidates:")
for p in decodeobj3_candidates + decodeobj_candidates:
    print(" ", p)

# -----------------------------
# Optional manual override examples
# -----------------------------
# Uncomment and edit these lines if the auto-selected files are not right.
#
# OBJ_PATH = SEARCH_FOLDER / "your_mask.obj"
# REFERENCE_MRI_PATH = SEARCH_FOLDER / "your_reference_mri.nii.gz"
# DECODEOBJ_M_DIR = SEARCH_FOLDER / "decodeOBJ_folder"
# DECODE_FUNCTION = "decodeOBJ3"
#
# OUT_MAT_PATH = OUTPUT_DIR / (OBJ_PATH.stem + "_decoded_from_matlab.mat")
# OUT_NIFTI_PATH = OUTPUT_DIR / (OBJ_PATH.stem + "_mask.nii.gz")
# OUT_QC_PNG = OUTPUT_DIR / (OBJ_PATH.stem + "_qc_overlay.png")

# 2. Inspect the `.obj` file type

This checks whether the file is:

- **Wavefront mesh OBJ**: ASCII text with `v` and `f` lines.
- **Analyze Object Map OBJ**: binary object map with embedded object names.

Your pasted sample looked like the second type.

In [ ]:
def read_first_bytes(path: Path, n: int = 4096) -> bytes:
    with open(path, "rb") as f:
        return f.read(n)


def classify_obj_file(path: Path) -> str:
    first = read_first_bytes(path, 4096)
    textish = first.decode("latin1", errors="ignore")
    wavefront_score = len(re.findall(r"(?m)^\s*[vf]\s+[-+0-9]", textish))
    has_binary_zero = b"\x00" in first
    has_analyze_names = any(name in textish for name in ["Original", "Object", ".Object"])

    if wavefront_score >= 5 and not has_binary_zero:
        return "wavefront_mesh_obj"
    if has_binary_zero or has_analyze_names:
        return "analyze_binary_object_map"
    return "unknown_obj_type"


def extract_printable_strings(path: Path, min_len: int = 4, max_strings: int = 100):
    data = path.read_bytes()
    strings = re.findall(rb"[\x20-\x7E]{%d,}" % min_len, data)
    return [s.decode("latin1", errors="ignore") for s in strings[:max_strings]]

obj_type = classify_obj_file(OBJ_PATH)
print("Detected OBJ type:", obj_type)
print("File size bytes:", OBJ_PATH.stat().st_size)
print("Printable strings found in file:")
for s in extract_printable_strings(OBJ_PATH, min_len=4, max_strings=40):
    print("  ", s)

# 3. Load the matching MRI reference

The output mask will copy the reference MRI's affine, orientation, voxel spacing, and header. This is what makes the mask usable in Python/MONAI/napari.

In [ ]:
ref_img = nib.load(str(REFERENCE_MRI_PATH))
ref_data = ref_img.get_fdata(dtype=np.float32)
ref_shape = ref_img.shape[:3]

print("Reference MRI shape:", ref_shape)
print("Reference MRI zooms:", ref_img.header.get_zooms()[:3])
print("Reference affine:")
print(ref_img.affine)

# 4. Decode Analyze Object Map with MATLAB

This cell writes a temporary MATLAB script that calls either:

```matlab
[objectmap, header] = decodeOBJ3('mask.obj')
```

or:

```matlab
[objectmap, header] = decodeOBJ('mask.obj')
```

Then MATLAB saves the decoded volume to a `.mat` file. Python loads that `.mat` file and saves a NIfTI mask using the reference MRI geometry.

If MATLAB cannot find `decodeOBJ3.m`, check `DECODEOBJ_M_DIR` and `DECODE_FUNCTION` above.

In [ ]:
def matlab_quote(path: Path) -> str:
    # MATLAB single-quoted string with escaped single quotes
    return str(path).replace("'", "''")


def find_matlab_executable(matlab_exe=None):
    if matlab_exe:
        p = Path(matlab_exe)
        if not p.exists():
            raise FileNotFoundError(f"MATLAB_EXE does not exist: {p}")
        return str(p)
    found = shutil.which("matlab")
    if found is None:
        raise FileNotFoundError(
            "Could not find 'matlab' on PATH. Set MATLAB_EXE to the full path to matlab.exe."
        )
    return found


def decode_analyze_obj_with_matlab(obj_path, out_mat_path, decode_m_dir, decode_function="decodeOBJ3", matlab_exe=None):
    obj_path = Path(obj_path).resolve()
    out_mat_path = Path(out_mat_path).resolve()
    decode_m_dir = Path(decode_m_dir).resolve()

    if not obj_path.exists():
        raise FileNotFoundError(obj_path)
    if not decode_m_dir.exists():
        raise FileNotFoundError(f"DECODEOBJ_M_DIR does not exist: {decode_m_dir}")

    # Check that the requested m-file exists.
    expected_m = decode_m_dir / f"{decode_function}.m"
    if not expected_m.exists():
        print(f"WARNING: Did not find {expected_m}")
        print("MATLAB may still find it if it is elsewhere on the MATLAB path.")

    matlab_script = f"""
    try
        addpath('{matlab_quote(decode_m_dir)}');
        obj_file = '{matlab_quote(obj_path)}';
        out_file = '{matlab_quote(out_mat_path)}';
        fprintf('Decoding Analyze OBJ: %s\\n', obj_file);
        [objectmap, header] = {decode_function}(obj_file);
        fprintf('Decoded objectmap size: ');
        disp(size(objectmap));
        save(out_file, 'objectmap', 'header', '-v7');
        fprintf('Saved MAT file: %s\\n', out_file);
    catch ME
        fprintf(2, 'MATLAB decode failed: %s\\n', ME.message);
        for k = 1:numel(ME.stack)
            fprintf(2, '  %s line %d\\n', ME.stack(k).file, ME.stack(k).line);
        end
        exit(1);
    end
    exit(0);
    """

    with tempfile.TemporaryDirectory() as td:
        script_path = Path(td) / "decode_analyze_obj_script.m"
        script_path.write_text(matlab_script, encoding="utf-8")
        matlab_cmd = find_matlab_executable(matlab_exe)
        cmd = [matlab_cmd, "-batch", f"run('{matlab_quote(script_path)}')"]
        print("Running:", " ".join(cmd))
        completed = subprocess.run(cmd, capture_output=True, text=True)
        print("MATLAB stdout:\n", completed.stdout)
        if completed.stderr.strip():
            print("MATLAB stderr:\n", completed.stderr)
        if completed.returncode != 0:
            raise RuntimeError(f"MATLAB failed with return code {completed.returncode}")

    if not out_mat_path.exists():
        raise FileNotFoundError(f"MATLAB completed but did not create {out_mat_path}")

    return out_mat_path

if obj_type == "analyze_binary_object_map":
    decoded_mat = decode_analyze_obj_with_matlab(
        OBJ_PATH,
        OUT_MAT_PATH,
        DECODEOBJ_M_DIR,
        decode_function=DECODE_FUNCTION,
        matlab_exe=MATLAB_EXE,
    )
    print("Decoded MAT:", decoded_mat)
else:
    print("Skipping MATLAB decode because this does not look like an Analyze binary object map.")

# 5. Convert decoded `.mat` object map to NIfTI

This cell loads the MATLAB-decoded `objectmap`, checks it against the reference MRI shape, applies optional orientation fixes if needed, and saves a NIfTI mask.

Most of the time, `AXES_ORDER = None` and `FLIP_AXES = ()` should be correct. If the overlay looks rotated/flipped, adjust these after viewing the QC image.

In [ ]:
# Optional fixes if the decoded objectmap orientation does not match the MRI.
# Examples:
# AXES_ORDER = (1, 0, 2)   # swap x and y
# FLIP_AXES = (0,)         # flip x-axis
AXES_ORDER = None
FLIP_AXES = ()

# If True, convert all nonzero labels to 1.
# If False, preserve separate object labels such as 1, 2, 3...
MAKE_BINARY_MASK = False


def load_objectmap_from_mat(mat_path: Path):
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    if "objectmap" not in mat:
        raise KeyError(f"No variable named 'objectmap' in {mat_path}. Variables: {list(mat.keys())}")
    arr = np.asarray(mat["objectmap"])
    return arr, mat


def orient_mask(mask, axes_order=None, flip_axes=()):
    out = np.asarray(mask)
    # Remove singleton dimensions if present, but keep 3D/4D if meaningful.
    out = np.squeeze(out)
    if out.ndim > 3:
        print("Decoded objectmap has >3 dimensions:", out.shape)
        print("Using the first volume along trailing dimensions. Edit this if you need another volume.")
        while out.ndim > 3:
            out = out[..., 0]
    if axes_order is not None:
        out = np.transpose(out, axes_order)
    for ax in flip_axes:
        out = np.flip(out, axis=ax)
    return out


def save_mask_as_nifti(mask, ref_img, out_path, make_binary=False):
    mask = np.asarray(mask)
    if make_binary:
        mask = (mask > 0).astype(np.uint8)
    else:
        # Keep integer labels but use a sane dtype.
        mask = mask.astype(np.int16)

    if mask.shape != ref_img.shape[:3]:
        raise ValueError(
            f"Mask shape {mask.shape} does not match reference shape {ref_img.shape[:3]}. "
            "Use AXES_ORDER / FLIP_AXES, or confirm the OBJ belongs to this MRI."
        )

    out_img = nib.Nifti1Image(mask, ref_img.affine, header=ref_img.header.copy())
    out_img.set_data_dtype(mask.dtype)
    nib.save(out_img, str(out_path))
    return out_path

if obj_type == "analyze_binary_object_map":
    raw_objmap, mat_dict = load_objectmap_from_mat(OUT_MAT_PATH)
    print("Raw decoded objectmap shape:", raw_objmap.shape)
    print("Raw dtype:", raw_objmap.dtype)
    print("Raw min/max:", np.nanmin(raw_objmap), np.nanmax(raw_objmap))

    mask = orient_mask(raw_objmap, axes_order=AXES_ORDER, flip_axes=FLIP_AXES)
    print("Oriented mask shape:", mask.shape)
    print("Unique labels:", np.unique(mask)[:50])

    save_mask_as_nifti(mask, ref_img, OUT_NIFTI_PATH, make_binary=MAKE_BINARY_MASK)
    print("Saved NIfTI mask:", OUT_NIFTI_PATH)
else:
    print("Skipping MAT-to-NIfTI because this was not decoded as an Analyze Object Map.")

# 6. QC overlay

This creates quick axial/coronal/sagittal overlays. Check that the mask is exactly where it should be.

If the mask is rotated or flipped, return to the previous cell and modify:

```python
AXES_ORDER = ...
FLIP_AXES = ...
```

In [ ]:
def robust_window(img, low=1, high=99):
    finite = img[np.isfinite(img)]
    vmin, vmax = np.percentile(finite, [low, high])
    return vmin, vmax


def pick_nonzero_center(mask):
    coords = np.argwhere(mask > 0)
    if coords.size == 0:
        return tuple(s // 2 for s in mask.shape)
    return tuple(np.round(coords.mean(axis=0)).astype(int))


def plot_qc_overlay(ref_img, mask_img_or_arr, out_png=None, alpha=0.35):
    img = ref_img.get_fdata(dtype=np.float32)
    if isinstance(mask_img_or_arr, nib.spatialimages.SpatialImage):
        mask_arr = mask_img_or_arr.get_fdata()
    else:
        mask_arr = np.asarray(mask_img_or_arr)

    center = pick_nonzero_center(mask_arr)
    x, y, z = center
    vmin, vmax = robust_window(img)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img[:, :, z].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
    axes[0].imshow(np.ma.masked_where(mask_arr[:, :, z].T <= 0, mask_arr[:, :, z].T), alpha=alpha, origin="lower")
    axes[0].set_title(f"Axial z={z}")

    axes[1].imshow(img[:, y, :].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
    axes[1].imshow(np.ma.masked_where(mask_arr[:, y, :].T <= 0, mask_arr[:, y, :].T), alpha=alpha, origin="lower")
    axes[1].set_title(f"Coronal y={y}")

    axes[2].imshow(img[x, :, :].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
    axes[2].imshow(np.ma.masked_where(mask_arr[x, :, :].T <= 0, mask_arr[x, :, :].T), alpha=alpha, origin="lower")
    axes[2].set_title(f"Sagittal x={x}")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()

    if out_png is not None:
        fig.savefig(out_png, dpi=160, bbox_inches="tight")
        print("Saved QC PNG:", out_png)
    plt.show()

if OUT_NIFTI_PATH.exists():
    mask_img = nib.load(str(OUT_NIFTI_PATH))
    plot_qc_overlay(ref_img, mask_img, out_png=OUT_QC_PNG)
else:
    print("No NIfTI mask found yet:", OUT_NIFTI_PATH)

# 7. Basic mask statistics

This gives a quick sanity check for label counts and approximate physical volumes.

In [ ]:
def mask_stats(mask_img):
    arr = np.asarray(mask_img.get_fdata())
    labels, counts = np.unique(arr.astype(np.int64), return_counts=True)
    voxel_volume = float(np.prod(mask_img.header.get_zooms()[:3]))
    rows = []
    for lab, count in zip(labels, counts):
        rows.append({
            "label": int(lab),
            "voxel_count": int(count),
            "physical_volume_units3": float(count * voxel_volume),
        })
    return pd.DataFrame(rows)

if OUT_NIFTI_PATH.exists():
    stats_df = mask_stats(nib.load(str(OUT_NIFTI_PATH)))
    display(stats_df)
    stats_csv = OUTPUT_DIR / (OBJ_PATH.stem + "_mask_stats.csv")
    stats_df.to_csv(stats_csv, index=False)
    print("Saved stats CSV:", stats_csv)

# 8. Batch conversion

Use this section when you have many `.obj` files. The table should have one row per mask:

```csv
obj_path,reference_mri_path,output_dir
Z:\data\mouse001.obj,Z:\data\mouse001.nii.gz,Z:\converted
Z:\data\mouse002.obj,Z:\data\mouse002.nii.gz,Z:\converted
```

The output for each row will be:

- decoded `.mat`
- mask `.nii.gz`
- QC overlay `.png`
- stats `.csv`

In [ ]:
BATCH_CSV = Path(r"Z:\path\to\batch_convert.csv")
RUN_BATCH = False


def convert_one_analyze_obj(obj_path, reference_mri_path, output_dir,
                            decode_m_dir, decode_function="decodeOBJ3",
                            matlab_exe=None, axes_order=None, flip_axes=(),
                            make_binary=False):
    obj_path = Path(obj_path)
    reference_mri_path = Path(reference_mri_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    ref = nib.load(str(reference_mri_path))
    mat_path = output_dir / f"{obj_path.stem}_decoded_from_matlab.mat"
    nifti_path = output_dir / f"{obj_path.stem}_mask.nii.gz"
    qc_path = output_dir / f"{obj_path.stem}_qc_overlay.png"
    stats_path = output_dir / f"{obj_path.stem}_mask_stats.csv"

    typ = classify_obj_file(obj_path)
    if typ != "analyze_binary_object_map":
        raise ValueError(f"Expected Analyze binary object map, got {typ}: {obj_path}")

    decode_analyze_obj_with_matlab(
        obj_path, mat_path, decode_m_dir,
        decode_function=decode_function,
        matlab_exe=matlab_exe,
    )
    raw, _ = load_objectmap_from_mat(mat_path)
    m = orient_mask(raw, axes_order=axes_order, flip_axes=flip_axes)
    save_mask_as_nifti(m, ref, nifti_path, make_binary=make_binary)

    m_img = nib.load(str(nifti_path))
    plot_qc_overlay(ref, m_img, out_png=qc_path)
    stats = mask_stats(m_img)
    stats.to_csv(stats_path, index=False)

    return {
        "obj_path": str(obj_path),
        "reference_mri_path": str(reference_mri_path),
        "nifti_mask_path": str(nifti_path),
        "qc_png_path": str(qc_path),
        "stats_csv_path": str(stats_path),
        "status": "ok",
    }

if RUN_BATCH:
    batch_df = pd.read_csv(BATCH_CSV)
    results = []
    for _, row in batch_df.iterrows():
        try:
            res = convert_one_analyze_obj(
                row["obj_path"],
                row["reference_mri_path"],
                row.get("output_dir", OUTPUT_DIR),
                DECODEOBJ_M_DIR,
                decode_function=DECODE_FUNCTION,
                matlab_exe=MATLAB_EXE,
                axes_order=AXES_ORDER,
                flip_axes=FLIP_AXES,
                make_binary=MAKE_BINARY_MASK,
            )
        except Exception as e:
            res = {
                "obj_path": row.get("obj_path", ""),
                "reference_mri_path": row.get("reference_mri_path", ""),
                "status": "failed",
                "error": repr(e),
            }
        results.append(res)

    results_df = pd.DataFrame(results)
    display(results_df)
    results_csv = OUTPUT_DIR / "batch_conversion_results.csv"
    results_df.to_csv(results_csv, index=False)
    print("Saved batch results:", results_csv)
else:
    print("Batch conversion disabled. Set RUN_BATCH = True after editing BATCH_CSV.")

# 9. Fallback: Wavefront mesh `.obj` conversion

This section is only for true Wavefront mesh files that contain text lines like:

```text
v 1.0 2.0 3.0
f 1 2 3
```

It will not work for Analyze binary Object Maps.

In [ ]:
def convert_wavefront_obj_mesh_to_nifti(obj_path, reference_mri_path, out_nifti_path,
                                        vertices_are_world=True, make_binary=True):
    import trimesh
    from scipy.ndimage import binary_fill_holes

    obj_path = Path(obj_path)
    ref = nib.load(str(reference_mri_path))
    ref_shape = ref.shape[:3]
    affine = ref.affine
    inv_affine = np.linalg.inv(affine)

    mesh = trimesh.load(str(obj_path), force="mesh")
    if mesh.is_empty:
        raise ValueError("Mesh loaded as empty. This is probably not a Wavefront mesh OBJ.")

    verts = np.asarray(mesh.vertices)
    if vertices_are_world:
        verts_vox = nib.affines.apply_affine(inv_affine, verts)
    else:
        verts_vox = verts

    mesh_vox = trimesh.Trimesh(vertices=verts_vox, faces=mesh.faces, process=False)
    vox = mesh_vox.voxelized(pitch=1.0).fill()

    mask = np.zeros(ref_shape, dtype=np.uint8)
    pts = np.round(vox.points).astype(int)
    valid = (
        (pts[:, 0] >= 0) & (pts[:, 0] < ref_shape[0]) &
        (pts[:, 1] >= 0) & (pts[:, 1] < ref_shape[1]) &
        (pts[:, 2] >= 0) & (pts[:, 2] < ref_shape[2])
    )
    pts = pts[valid]
    mask[pts[:, 0], pts[:, 1], pts[:, 2]] = 1
    mask = binary_fill_holes(mask).astype(np.uint8)

    if make_binary:
        mask = (mask > 0).astype(np.uint8)

    out_img = nib.Nifti1Image(mask, ref.affine, header=ref.header.copy())
    out_img.set_data_dtype(mask.dtype)
    nib.save(out_img, str(out_nifti_path))
    return out_nifti_path

if obj_type == "wavefront_mesh_obj":
    convert_wavefront_obj_mesh_to_nifti(OBJ_PATH, REFERENCE_MRI_PATH, OUT_NIFTI_PATH)
    print("Saved mesh-derived NIfTI mask:", OUT_NIFTI_PATH)
else:
    print("Skipping Wavefront conversion because detected OBJ type is:", obj_type)

# 10. Troubleshooting

## MATLAB says `decodeOBJ3` is undefined

Check that `DECODEOBJ_M_DIR` points to the folder containing `decodeOBJ3.m`. Also try:

```python
DECODE_FUNCTION = "decodeOBJ"
```

if you downloaded the older reader.

## Mask shape does not match MRI shape

Possibilities:

1. The `.obj` file does not belong to this MRI.
2. The reference MRI has been resampled/cropped since the mask was created.
3. The decoded object map needs a transpose or flip.

Try changing:

```python
AXES_ORDER = (1, 0, 2)
FLIP_AXES = (0,)
```

or other combinations, then rerun the MAT-to-NIfTI and QC cells.

## Overlay appears shifted, not just flipped

That usually means the mask was created on a different image grid than the reference MRI. Use the exact MRI volume that was open in Analyze when the object map was created.

## Multiple labels are present

That is good. The Analyze object file can contain multiple objects. Keep `MAKE_BINARY_MASK = False` if you want separate labels. Set it to `True` only if all objects should become one ROI.

## Text editor shows strange characters

That is expected for an Analyze binary object map. Do not edit or save the file in a text editor, because that can corrupt it.